# CS781 Capstone -- Phase 7 re-ranking: lexical model + Phase 1 dense models + pool-aware re-ranking

No new training. This notebook starts from the experiments you already ran and works only on **score matrices** that are quick to recompute and evaluate on dev:

1. **Pool diagnostics** - how the candidate pools are built (are the queries themselves in the pool? is every pool document relevant to some query?).
2. **Lexical retrieval on the full case text** (the earlier TF-IDF only saw the first 30,000 characters): TF-IDF and BM25 variants.
3. **Dense scores from your trained Phase 1 models with more of each case** (up to 8 chunks instead of 2); no retraining.
4. **Fusion and re-ranking** tuned on dev: dense/lexical weight, removing other queries from the candidates, pseudo-relevance feedback, and a dual softmax that lets every candidate compete across queries.
5. Writes two submissions: the best overall pipeline, and a simpler one that does not use the other test queries.

**Please read before submitting:** the dual softmax (and, to a lesser extent, query removal) uses properties of how the pools are built and the other test queries. It is a legitimate ranking strategy on the data you were given, but state it plainly in the report and check with the TAs that it is acceptable. The `simple` submission avoids the cross-query step.

**Inputs to attach:** the four Phase 1 model datasets and the outputs of the two earlier Phase 7 notebook runs (experiments 1-4 and 5-6), so the `best_model.pt` files can be found.

## 1. Hugging Face login

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF2")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Configuration

In [3]:
import os, random, json, time, math, zipfile, shutil, re, gc, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PCR_FILES = {
    "dev":  ("Phase_7_PCR/Dev/dev.json",   "Phase_7_PCR/Dev/dev_candidates.json"),
    "test": ("Phase_7_PCR/Test/test.json", "Phase_7_PCR/Test/test_candidates.json"),
}
MODEL_PATHS = {
    "gpt2":     "/kaggle/input/datasets/himangshumahato/model1",
    "swiglu":   "/kaggle/input/datasets/himangshumahato/model2",
    "gqa":      "/kaggle/input/datasets/himangshumahato/model3",
    "prefixlm": "/kaggle/input/datasets/himangshumahato/model4",
}
MODEL_CONTEXT_LENGTH = 1024
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 768, 12, 12, 4

# ---- representation: MORE of each case than during training (the dense models were trained on 2 chunks) ----
CHUNK_LEN, MAX_CHUNKS, MIN_CHUNK = 384, 8, 32
HEAD_CHARS = 40000                  # raw text tokenized per case (about 8 chunks)
TEXT_LIMIT = 300000                 # characters kept per case; lexical models use all of it
PROJ_DIM, TAU, TOP_K, MAX_PREFIX = 512, 0.05, 7, 32
HYBRID_ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
BASE_OUTPUT_DIR = "/kaggle/working/cs781_phase7_rerank"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def experiment_dirs(name):
    d = os.path.join(BASE_OUTPUT_DIR, name)
    os.makedirs(d, exist_ok=True)
    return {"dir": d, "model_path": os.path.join(d, "best_model.pt"), "run_info_path": os.path.join(d, "run_info.json"),
            "dense_dir": d, "hybrid_dir": os.path.join(d, "hybrid")}
print("Configuration loaded.")

Configuration loaded.


## 3. Tokenizer

In [4]:
%%writefile a1_tokenizer.py
"""From-scratch byte-level BPE tokenizer (GPT-2 style), no external tokenizer libs.

Differences from the plain-bytes version:
  * Text is first split into "words" with a GPT-2 style regex, and merges never
    cross word boundaries. Training and encoding then work on *unique words*
    (with counts) instead of the whole corpus, which is orders of magnitude faster.
  * Training uses incremental pair counts + a lazy max-heap (no full rescan per merge).
  * Encoding applies merges per word by rank, with a word -> ids cache.
  * One special token, <|endoftext|>, used as EOS / PAD / BOS.

Interface:
    tok.encode(text, add_eos=False) -> list[int]
    tok.decode(ids, skip_special_tokens=True) -> str
    tok.vocab_size / tok.get_vocab_size()
    tok.eos_token_id, tok.pad_token_id, tok.bos_token_id
    tok.save(path) / ByteLevelBPE.load(path)
"""

from __future__ import annotations

import heapq
import json
import time
from collections import Counter, defaultdict

import regex as re  # `pip install regex` (needed for \p{L} / \p{N})

# GPT-2 pre-tokenization pattern
_PAT = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

EOS_TOKEN = "<|endoftext|>"


class ByteLevelBPE:
    def __init__(self):
        # ids 0-255 = raw bytes; 256.. = learned merges; last id = <|endoftext|>
        self.merges: dict[tuple[int, int], int] = {}      # (a, b) -> new_id (id order == rank order)
        self.itos: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.eos_token_id = 256
        self.vocab_size = 257
        self._cache: dict[str, tuple] = {}

    # alias attributes so the tokenizer plugs into HF-style code
    @property
    def pad_token_id(self): return self.eos_token_id
    @property
    def bos_token_id(self): return self.eos_token_id

    def __len__(self): return self.vocab_size
    def get_vocab_size(self): return self.vocab_size

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        """Learn (vocab_size - 257) merges; the final id is reserved for <|endoftext|>."""
        if vocab_size < 257:
            raise ValueError("vocab_size must be at least 257 (256 bytes + EOS)")
        num_merges = vocab_size - 257
        t0 = time.time()

        # 1) pre-tokenize and count unique words
        word_counts = Counter(_PAT.findall(text))
        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        if verbose:
            print(f"[bpe] {len(words):,} unique words from {sum(freqs):,} total words")

        # 2) initial pair statistics
        stats: dict[tuple[int, int], int] = defaultdict(int)
        where: dict[tuple[int, int], set] = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for p in zip(w, w[1:]):
                stats[p] += f
                where[p].add(wi)

        # lazy max-heap: (-count, a, b) -> highest count first, ties -> smallest (a, b)
        heap = [(-c, a, b) for (a, b), c in stats.items()]
        heapq.heapify(heap)

        self.merges, self.itos = {}, {i: bytes([i]) for i in range(256)}

        # 3) merge loop
        for i in range(num_merges):
            pair, count = None, 0
            while heap:
                negc, a, b = heapq.heappop(heap)
                if stats.get((a, b), 0) == -negc:   # skip stale heap entries
                    pair, count = (a, b), -negc
                    break
            if pair is None or count < 2:
                break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.itos[new_id] = self.itos[pair[0]] + self.itos[pair[1]]

            touched = set()
            for wi in list(where[pair]):
                w, f = words[wi], freqs[wi]
                # remove this word's old pair contributions
                for p in zip(w, w[1:]):
                    stats[p] -= f
                    where[p].discard(wi)
                    touched.add(p)
                # apply merge (leftmost, non-overlapping)
                a, b = pair
                out, j, n = [], 0, len(w)
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2
                    else:
                        out.append(w[j]); j += 1
                words[wi] = out
                # add new pair contributions
                for p in zip(out, out[1:]):
                    stats[p] += f
                    where[p].add(wi)
                    touched.add(p)

            for p in touched:
                c = stats.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p[0], p[1]))
                else:
                    stats.pop(p, None)
                    where.pop(p, None)

            if verbose and (i % 1000 == 0 or i == num_merges - 1):
                print(f"[bpe] merge {i + 1}/{num_merges} | pair={pair} count={count:,} "
                      f"| elapsed={time.time() - t0:.1f}s")

        self.eos_token_id = 256 + len(self.merges)
        self.vocab_size = self.eos_token_id + 1
        self._cache = {}

    # Inference
    def _encode_word(self, word: str) -> tuple:
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = list(word.encode("utf-8"))
        merges = self.merges
        while len(ids) > 1:
            # pick the applicable merge learned earliest (lowest id)
            best, best_id = None, None
            for p in zip(ids, ids[1:]):
                nid = merges.get(p)
                if nid is not None and (best_id is None or nid < best_id):
                    best, best_id = p, nid
            if best is None:
                break
            a, b = best
            out, j, n = [], 0, len(ids)
            while j < n:
                if j < n - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(best_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        res = tuple(ids)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = res
        return res

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids: list[int] = []
        for w in _PAT.findall(text):
            ids.extend(self._encode_word(w))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        for i in ids:
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(EOS_TOKEN.encode("utf-8"))
                continue
            parts.append(self.itos[i])
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        data = {
            "merges": [[a, b, nid] for (a, b), nid in self.merges.items()],
            "vocab_size": self.vocab_size,
            "eos_token_id": self.eos_token_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str) -> "ByteLevelBPE":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        for a, b, nid in data["merges"]:
            tok.merges[(a, b)] = nid
            tok.itos[nid] = tok.itos[a] + tok.itos[b]
        tok.eos_token_id = data["eos_token_id"]
        tok.vocab_size = data["vocab_size"]
        return tok


def train_tokenizer(train_text: str, vocab_size: int):
    """Train a byte-level BPE tokenizer on `train_text` and return it."""
    tok = ByteLevelBPE()
    tok.train(train_text, vocab_size, verbose=True)
    return tok

Writing a1_tokenizer.py


In [5]:
from a1_tokenizer import ByteLevelBPE, _PAT
tokenizer = ByteLevelBPE.load(os.path.join(MODEL_PATHS["gqa"], "bpe_tokenizer.json"))
print("Vocabulary size:", len(tokenizer))
EOS_ID = tokenizer.eos_token_id
PAD_ID = EOS_ID

Vocabulary size: 32000


## 4. Data (dev and test) and metrics

In [6]:
ID_KEYS   = ["id", "case_id", "doc_id", "document_id"]
TEXT_KEYS = ["text", "document", "case_text", "sentences"]
REL_KEYS  = ["relevant_candidates", "relevant", "cited_cases", "labels"]

def _get(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None

def doc_text(x, limit):
    """Case text (a string or a list of sentences) -> one string of at most `limit` characters."""
    if isinstance(x, str):
        return x[:limit]
    if isinstance(x, (list, tuple)):
        out, n = [], 0
        for s in x:
            s = s if isinstance(s, str) else str(s)
            out.append(s); n += len(s) + 1
            if n >= limit:
                break
        return " ".join(out)[:limit]
    return str(x)[:limit]

def _items(raw):
    return list(raw.items()) if isinstance(raw, dict) else [(None, v) for v in raw]

def to_pool(raw, limit):
    """Candidate pool -> (ids, texts). Accepts a list of dicts or a dict id -> dict / text."""
    ids, texts = [], []
    for k, v in _items(raw):
        if isinstance(v, dict):
            rid = _get(v, ID_KEYS); rid = k if rid is None else rid
            txt = _get(v, TEXT_KEYS)
            if txt is None:
                raise KeyError(f"no text field in a candidate; keys: {list(v.keys())}")
        else:
            rid, txt = k, v
        if rid is None:
            raise KeyError("candidate without an id")
        ids.append(str(rid)); texts.append(doc_text(txt, limit))
    return ids, texts

def to_queries(raw, limit, with_labels):
    out = []
    for k, v in _items(raw):
        if not isinstance(v, dict):
            raise ValueError("a query record must be a dict")
        qid = _get(v, ID_KEYS); qid = k if qid is None else qid
        txt = _get(v, TEXT_KEYS)
        if txt is None:
            raise KeyError(f"no text field in a query; keys: {list(v.keys())}")
        rel = _get(v, REL_KEYS) if with_labels else []
        out.append({"id": str(qid), "text": doc_text(txt, limit), "rel": [str(r) for r in (rel or [])]})
    return out

In [8]:
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def hf_json(path):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def chunk_doc(text):
    """Head of the case -> up to MAX_CHUNKS chunks of CHUNK_LEN token ids (a short trailing chunk is dropped)."""
    ids = tokenizer.encode(text[:HEAD_CHARS])[:CHUNK_LEN * MAX_CHUNKS]
    chunks = [ids[i:i + CHUNK_LEN] for i in range(0, len(ids), CHUNK_LEN)]
    if len(chunks) > 1 and len(chunks[-1]) < MIN_CHUNK:
        chunks.pop()
    return chunks or [[EOS_ID]]


def tfidf_score_matrix(q_texts, pool_texts):
    """Lexical baseline / hard-negative miner: cosine similarity of TF-IDF vectors, [n_queries, n_pool]."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, max_features=300000, dtype=np.float32)
    vec.fit(pool_texts + q_texts)
    Xq, Xc = vec.transform(q_texts), vec.transform(pool_texts)
    return (Xq @ Xc.T).toarray().astype(np.float32)


class Split:
    """Queries + their own candidate pool (a split's queries must only retrieve from its pool)."""
    def __init__(self, name, q_raw, pool_raw, with_labels):
        self.name = name
        self.pool_ids, self.pool_texts = to_pool(pool_raw, TEXT_LIMIT)
        self.pool_index = {d: i for i, d in enumerate(self.pool_ids)}
        self.queries = to_queries(q_raw, TEXT_LIMIT, with_labels)
        missing = 0
        for q in self.queries:
            q["rel_idx"] = [self.pool_index[r] for r in q["rel"] if r in self.pool_index]
            missing += len(q["rel"]) - len(q["rel_idx"])
            q["self_idx"] = self.pool_index.get(q["id"], -1)      # the query itself must never be returned
        self.n_missing_gold = missing
        self._q_chunks = self._pool_chunks = self._tfidf = None

    @property
    def q_chunks(self):
        if self._q_chunks is None:
            self._q_chunks = [chunk_doc(q["text"]) for q in tqdm(self.queries, desc=f"{self.name} queries")]
        return self._q_chunks

    @property
    def pool_chunks(self):
        if self._pool_chunks is None:
            self._pool_chunks = [chunk_doc(t) for t in tqdm(self.pool_texts, desc=f"{self.name} pool")]
        return self._pool_chunks

    def tfidf_scores(self):
        if self._tfidf is None:
            self._tfidf = tfidf_score_matrix([q["text"] for q in self.queries], self.pool_texts)
        return self._tfidf

    def gold(self):
        return {q["id"]: set(q["rel"]) for q in self.queries}


splits = {}
for name, (qf, cf) in PCR_FILES.items():
    splits[name] = Split(name, hf_json(qf), hf_json(cf), with_labels=(name != "test"))
    s = splits[name]
    n_gold = [len(q["rel"]) for q in s.queries]
    print(f"{name:5s}: {len(s.queries):4d} queries | pool {len(s.pool_ids):5d} | "
          + (f"gold per query mean {np.mean(n_gold):.1f} (min {min(n_gold)}, max {max(n_gold)}) | gold missing from pool: {s.n_missing_gold}" if name != "test" else "no labels"))
    assert len(set(s.pool_ids)) == len(s.pool_ids), "duplicate candidate ids"
dev_split, test_split = splits["dev"], splits["test"]
print("query ids that also appear in their own pool (dev/test):", [sum(q["self_idx"] >= 0 for q in splits[n].queries) for n in ("dev", "test")])

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

Phase_7_PCR/Dev/dev_candidates.json: downloading bytes:           |  0.00B            

dev  :  118 queries | pool  1023 | gold per query mean 8.2 (min 1, max 92) | gold missing from pool: 0


test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test :  237 queries | pool  1727 | no labels
query ids that also appear in their own pool (dev/test): [118, 237]


In [9]:
def f1_at_k(preds, golds, k=TOP_K):
    """preds: {qid: [ids]}; golds: {qid: set(ids)} -> dict(p, r, f1), micro-averaged over queries."""
    tp = n_pred = n_gold = 0
    for q, g in golds.items():
        p = list(preds.get(q, []))[:k]
        tp += len(set(p) & g); n_pred += len(p); n_gold += len(g)
    P = tp / n_pred if n_pred else 0.0
    R = tp / n_gold if n_gold else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return {"p": P, "r": R, "f1": F}

def oracle_f1_at_k(golds, k=TOP_K):
    tp = sum(min(k, len(g)) for g in golds.values())
    n_pred, n_gold = k * len(golds), sum(len(g) for g in golds.values())
    P, R = tp / n_pred, tp / n_gold
    return {"p": P, "r": R, "f1": 2 * P * R / (P + R)}

def zrow(m):
    return (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-6)

def hybrid_scores(dense, tfidf, alpha):
    """alpha * z(dense) + (1 - alpha) * z(tfidf); alpha = 1 keeps the dense ranking."""
    return alpha * zrow(dense) + (1 - alpha) * zrow(tfidf)

def topk_ids(score_row, ids, exclude_pos=-1, k=TOP_K):
    s = np.array(score_row, dtype=np.float64)
    if exclude_pos >= 0:
        s[exclude_pos] = -1e18
    if len(s) <= k:
        idx = np.argsort(-s)
    else:
        idx = np.argpartition(-s, k)[:k]
        idx = idx[np.argsort(-s[idx])]
    return [ids[i] for i in idx[:k]]

def predictions_from_scores(split, scores, pool_idx=None):
    """scores: [n_queries, len(pool_idx or full pool)] -> {query id: top-7 candidate ids}."""
    pool_idx = list(range(len(split.pool_ids))) if pool_idx is None else pool_idx
    ids = [split.pool_ids[i] for i in pool_idx]
    pos = {p: i for i, p in enumerate(pool_idx)}
    return {q["id"]: topk_ids(scores[qi], ids, pos.get(q["self_idx"], -1)) for qi, q in enumerate(split.queries)}

def eval_scores(split, scores, pool_idx=None):
    return f1_at_k(predictions_from_scores(split, scores, pool_idx), split.gold())

def make_small_pool(split, cap, seed=0):
    """All gold candidates of the split's queries plus random others, up to `cap` documents."""
    gold = {i for q in split.queries for i in q["rel_idx"]}
    others = [i for i in range(len(split.pool_ids)) if i not in gold]
    np.random.RandomState(seed).shuffle(others)
    return sorted(list(gold) + others[:max(0, cap - len(gold))])

def tune_hybrid(split, dense, tfidf, pool_idx=None):
    tf = tfidf if pool_idx is None else tfidf[:, pool_idx]
    best = None
    for a in HYBRID_ALPHAS:
        r = eval_scores(split, hybrid_scores(dense, tf, a), pool_idx)
        if best is None or r["f1"] >= best[1]["f1"] - 1e-12 and (r["f1"] > best[1]["f1"] or a > best[0]):
            best = (a, r)
    return best

def fmt(r):
    return f"P@7 {r['p']:.4f} | R@7 {r['r']:.4f} | F1@7 {r['f1']:.4f}"

def validate_predictions(preds, split):
    pool = set(split.pool_ids)
    assert set(preds) == {q["id"] for q in split.queries}, "predictions must cover every query"
    for qid, ids in preds.items():
        assert len(ids) == TOP_K and len(set(ids)) == TOP_K, f"{qid}: need {TOP_K} unique ids"
        assert all(i in pool for i in ids), f"{qid}: id outside the candidate pool"

def write_submission(out_dir, preds, split):
    validate_predictions(preds, split)
    os.makedirs(out_dir, exist_ok=True)
    pj, zp = os.path.join(out_dir, "prediction.json"), os.path.join(out_dir, "submission.zip")
    with open(pj, "w") as f:
        json.dump(preds, f)
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    print("saved:", zp, "| queries:", len(preds))

## 5. Phase 1 backbones, case encoder and the adaptation layers (needed to rebuild the trained models)

In [10]:
import math as _m

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd); self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * n_embd, n_embd); self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2ScratchLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([TransformerBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class SwiGLUMLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, hidden_mult=4):
        super().__init__()
        hidden = int(n_embd * hidden_mult * 2 / 3)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class CausalSelfAttentionRMS(CausalSelfAttention):
    pass


class RMSSwiGLUBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd); self.attn = CausalSelfAttentionRMS(n_embd, n_head, n_positions, dropout)
        self.ln_2 = RMSNorm(n_embd); self.mlp = SwiGLUMLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class RMSSwiGLULM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([RMSSwiGLUBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head; self.num_groups = n_head // n_kv_head
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        k = k.repeat_interleave(self.num_groups, dim=1); v = v.repeat_interleave(self.num_groups, dim=1)
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class GQATransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'),
                                 nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class ScratchGroupedQueryLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class PrefixLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

print("Backbone classes defined.")

Backbone classes defined.


In [11]:
from safetensors.torch import load_file

BACKBONE_REGISTRY = {
    "gpt2":     {"cls": GPT2ScratchLM,         "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "swiglu":   {"cls": RMSSwiGLULM,           "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "gqa":      {"cls": ScratchGroupedQueryLM, "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
    "prefixlm": {"cls": PrefixLM,              "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
}

def load_lm(name):
    spec = BACKBONE_REGISTRY[name]
    model = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **spec["kwargs"])
    state = load_file(os.path.join(MODEL_PATHS[name], "model.safetensors"))
    print(f"[{name}] load_state_dict:", model.load_state_dict(state))
    return model

def build_student_lm(backbone):
    """Half the layers (0,2,4,...) of a Phase 1 backbone, initialised from those Phase 1 layers."""
    reg = BACKBONE_REGISTRY[backbone]
    keep = list(range(0, N_LAYER, 2))
    kwargs = dict(reg["kwargs"]); kwargs["n_layer"] = len(keep)
    lm = reg["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **kwargs)
    state = load_file(os.path.join(MODEL_PATHS[backbone], "model.safetensors"))
    new = {}
    for k, v in state.items():
        if k.startswith("h."):
            parts = k.split(".")
            if int(parts[1]) in keep:
                parts[1] = str(keep.index(int(parts[1])))
                new[".".join(parts)] = v
        else:
            new[k] = v
    print("[student] load_state_dict:", lm.load_state_dict(new))
    return lm

In [12]:
from torch.utils.checkpoint import checkpoint

class DocEncoder(nn.Module):
    def __init__(self, lm, n_embd=N_EMBD, proj_dim=PROJ_DIM, n_prefix=0, raw=False):
        super().__init__()
        if hasattr(lm, "lm_head"):
            del lm.lm_head                      # retrieval does not predict tokens
        self.lm, self.raw, self.n_prefix = lm, raw, n_prefix
        self.proj = nn.Linear(2 * n_embd, proj_dim)
        self.prefix_k = self.prefix_v = None
        if n_prefix > 0:
            attn = lm.h[0].attn
            n_kv = getattr(attn, "n_kv_head", attn.n_head)
            shape = (len(lm.h), n_kv, n_prefix, attn.head_dim)
            self.prefix_k = nn.Parameter(torch.randn(shape) * 0.02)
            self.prefix_v = nn.Parameter(torch.randn(shape) * 0.02)

    def _prefix_past(self, B):
        dt = torch.float16 if (torch.is_autocast_enabled() and torch.cuda.is_available()) else torch.float32
        return [(self.prefix_k[i].unsqueeze(0).expand(B, -1, -1, -1).to(dt),
                 self.prefix_v[i].unsqueeze(0).expand(B, -1, -1, -1).to(dt)) for i in range(self.prefix_k.size(0))]

    def hidden(self, ids):
        B, T = ids.shape
        past = self._prefix_past(B) if self.prefix_k is not None else None
        x = self.lm.wte(ids) + self.lm.wpe(torch.arange(T, device=ids.device))
        x = self.lm.drop(x)
        for i, block in enumerate(self.lm.h):
            lp = None if past is None else past[i]
            if getattr(self, "grad_ckpt", False) and self.training and torch.is_grad_enabled():
                # recompute this block's activations in the backward pass instead of storing them (less memory, ~30% slower)
                x = checkpoint(lambda h, b=block, p=lp: b(h, layer_past=p)[0], x, use_reentrant=False)
            else:
                x, _ = block(x, layer_past=lp)
        return self.lm.ln_f(x)

    def forward_chunks(self, ids, lengths):
        """ids [N, L] (right-padded), lengths [N] -> L2-normalised chunk embeddings [N, D]."""
        h = self.hidden(ids)
        N, L = ids.shape
        mask = (torch.arange(L, device=ids.device)[None, :] < lengths[:, None]).float().unsqueeze(-1)
        mean = (h.float() * mask).sum(1) / mask.sum(1).clamp(min=1)
        if self.raw:
            emb = mean
        else:
            last = h[torch.arange(N, device=ids.device), lengths - 1].float()
            emb = self.proj(torch.cat([last, mean], dim=-1))
        return F.normalize(emb, dim=-1)

print("DocEncoder defined.")

DocEncoder defined.


In [13]:
def pad_chunks(chunks):
    L = max(len(c) for c in chunks)
    ids = torch.full((len(chunks), L), PAD_ID, dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.tensor(c)
    return ids, torch.tensor([len(c) for c in chunks])

def encode_docs_train(model, docs_chunks):
    """Case embeddings WITH gradients: mean of the chunk embeddings, re-normalised."""
    flat = [c for ch in docs_chunks for c in ch]
    counts = [len(ch) for ch in docs_chunks]
    ids, lengths = pad_chunks(flat)
    emb = model.forward_chunks(ids.to(device), lengths.to(device))
    parts = torch.split(emb, counts)
    return F.normalize(torch.stack([p.mean(0) for p in parts]), dim=-1)

@torch.no_grad()
def embed_chunks_all(model, docs_chunks, bs=32):
    """Chunk embeddings of many cases, no gradients. Returns (emb [n_chunks, D] float32, counts per case)."""
    model.eval()
    flat = [c for ch in docs_chunks for c in ch]
    order = sorted(range(len(flat)), key=lambda k: len(flat[k]))
    out = [None] * len(flat)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        ids, lengths = pad_chunks([flat[k] for k in idx])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            e = model.forward_chunks(ids.to(device), lengths.to(device))
        for j, k in enumerate(idx):
            out[k] = e[j].float()
    return torch.stack(out), [len(ch) for ch in docs_chunks]

def aggregate(emb, counts, mode="mean"):
    parts = torch.split(emb, counts)
    docs = torch.stack([p.mean(0) for p in parts]) if mode == "mean" else torch.stack([p[0] for p in parts])
    return F.normalize(docs, dim=-1)

def maxsim_scores(q_emb, q_counts, c_emb, c_counts):
    """Late interaction: for each query chunk the best-matching candidate chunk, averaged over the query's chunks."""
    S = q_emb @ c_emb.T
    cols = torch.stack([b.max(dim=1).values for b in torch.split(S, c_counts, dim=1)], dim=1)
    return torch.stack([b.mean(0) for b in torch.split(cols, q_counts, dim=0)])

@torch.no_grad()
def dense_scores(model, split, pool_idx=None, mode="mean"):
    """Cosine scores [n_queries, len(pool)] (numpy). Returns (scores, pool_idx used)."""
    pool_idx = list(range(len(split.pool_ids))) if pool_idx is None else pool_idx
    q_emb, qc = embed_chunks_all(model, split.q_chunks)
    c_emb, cc = embed_chunks_all(model, [split.pool_chunks[i] for i in pool_idx])
    if mode == "maxsim":
        S = maxsim_scores(q_emb, qc, c_emb, cc)
    else:
        S = aggregate(q_emb, qc, mode) @ aggregate(c_emb, cc, mode).T
    return S.cpu().numpy(), pool_idx

In [14]:
class LoRMALinear(nn.Module):
    """LoRMA: W' = (I + (alpha/r) B A) W0 (multiplicative low-rank update)."""
    def __init__(self, base_linear, r=8, alpha=16):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base_linear.out_features
        self.r = r; self.alpha = alpha
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.A = nn.Parameter(torch.randn(r, d_out) * 0.01)
        self.register_buffer("eye", torch.eye(d_out))

    def forward(self, x):
        M = self.eye + (self.alpha / self.r) * (self.B @ self.A)
        return F.linear(x, M @ self.base.weight, self.base.bias)

def apply_lorma(model, target_names, r=8, alpha=16):
    for parent_name, parent in list(model.named_modules()):
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_names:
                setattr(parent, child_name, LoRMALinear(child, r=r, alpha=alpha))
    return model


class IA3QKV(nn.Module):
    """IA3 on the fused q/k/v projection: keys and values are rescaled by learned vectors (initialised to 1)."""
    def __init__(self, base, n_embd):
        super().__init__()
        self.base, self.n_embd = base, n_embd
        for p in self.base.parameters():
            p.requires_grad = False
        self.l_k = nn.Parameter(torch.ones(n_embd))
        self.l_v = nn.Parameter(torch.ones(n_embd))

    def forward(self, x):
        q, k, v = self.base(x).split(self.n_embd, dim=-1)
        return torch.cat([q, k * self.l_k, v * self.l_v], dim=-1)

class IA3Input(nn.Module):
    """IA3 on the MLP: the intermediate activation is rescaled before the output projection."""
    def __init__(self, base, dim):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.l_ff = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return self.base(x * self.l_ff)

def apply_ia3(lm, n_embd):
    """For the GPT-2 style blocks (fused c_attn, MLP with c_proj)."""
    for blk in lm.h:
        blk.attn.c_attn = IA3QKV(blk.attn.c_attn, n_embd)
        blk.mlp.c_proj = IA3Input(blk.mlp.c_proj, 4 * n_embd)
    return lm

print("LoRMA and IA3 defined. (Prefix Tuning is inside DocEncoder.)")

LoRMA and IA3 defined. (Prefix Tuning is inside DocEncoder.)


In [17]:
def build_model_for(spec):
    """Phase 1 backbone + one adaptation technique + the projection head. Returns (model, [(params, lr)])."""
    lm = load_lm(spec["backbone"])
    for p in lm.parameters():
        p.requires_grad = False
    tech, n_prefix = spec["technique"], 0
    if tech == "lorma":
        lm = apply_lorma(lm, spec["targets"], r=spec["r"], alpha=spec["alpha"])
    elif tech == "ia3":
        lm = apply_ia3(lm, N_EMBD)
    elif tech == "partial":
        for blk in lm.h[-spec["n_blocks"]:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in lm.ln_f.parameters():
            p.requires_grad = True
    elif tech == "full":
        for p in lm.parameters():
            p.requires_grad = True
    elif tech == "prefix":
        n_prefix = spec["n_soft"]
        assert n_prefix <= MAX_PREFIX
    else:
        raise ValueError(tech)
    model = DocEncoder(lm, n_prefix=n_prefix).to(device)
    model.grad_ckpt = spec.get("grad_ckpt", False)
    lm_params = [p for p in model.lm.parameters() if p.requires_grad]
    extra = [model.prefix_k, model.prefix_v] if n_prefix else []
    groups = [(list(model.proj.parameters()), spec.get("lr_head", 1e-3))]
    if lm_params or extra:
        groups.append((lm_params + extra, spec["lr"]))
    return model, groups

def build_student(spec):
    model = DocEncoder(build_student_lm(spec["backbone"])).to(device)
    model.grad_ckpt = spec.get("grad_ckpt", False)
    for p in model.parameters():
        p.requires_grad = True
    return model, [(list(model.proj.parameters()), spec.get("lr_head", 1e-3)),
                   (list(model.lm.parameters()), spec["lr"])]

def dev_f1_of(info_path):
    return json.load(open(info_path)).get("dev", {}).get("dense", {}).get("f1", -1)

def pick_best_spec():
    """The best finished experiment among e1..e5 by DENSE dev F1@7 (the primary metric), or TEACHER_EXP."""
    specs = {k: v for k, v in EXPERIMENT_SPECS.items() if k != "e6"}
    if TEACHER_EXP:
        return specs[TEACHER_EXP]
    best, best_spec = -1.0, None
    for k, s in specs.items():
        p = experiment_dirs(s["name"])["run_info_path"]
        if os.path.exists(p) and dev_f1_of(p) > best:
            best, best_spec = dev_f1_of(p), s
    assert best_spec is not None, "no finished experiment found: run e1-e5 first or set TEACHER_EXP / TEACHER_STATE_PATH"
    print("best experiment:", best_spec["name"], f"(dev F1@7 {best:.4f})")
    return best_spec

# def find_checkpoint(spec):
#     """This session's checkpoint if it exists, otherwise the one in an attached earlier notebook output."""
#     own = experiment_dirs(spec["name"])["model_path"]
#     if os.path.exists(own):
#         return own
#     import glob
#     hits = glob.glob(f"/kaggle/input/**/{spec['name']}/best_model.pt", recursive=True)
#     assert hits, f"no checkpoint for {spec['name']}: attach the earlier notebook output as an input"
#     return hits[0]
def find_checkpoint(spec):
    """Finds checkpoint by searching recursively through any nested subfolders."""
    own = experiment_dirs(spec["name"])["model_path"]
    if os.path.exists(own):
        return own
    import glob
    # Searching with recursive double-asterisk across all input subdirectories
    hits = glob.glob(f"/kaggle/input/**/{spec['name']}/best_model.pt", recursive=True)
    if not hits:
        # Fallback search if nested under working directory or alternative folder structures
        hits = glob.glob(f"/kaggle/**/{spec['name']}/best_model.pt", recursive=True)
    assert hits, f"no checkpoint for {spec['name']}: check attached dataset structure"
    return hits[0]

def load_experiment_model(spec, path=None):
    model, _ = build_model_for(spec)
    load_trainable(model, path or find_checkpoint(spec))
    return model

def run_experiment(spec):
    name = spec["name"]; paths = experiment_dirs(name)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    print(f"===== {name} =====")
    teacher, teacher_name = None, None
    if spec["technique"] == "kd":
        t_spec = pick_best_spec(); teacher_name = t_spec["name"]
        teacher = load_experiment_model(t_spec, TEACHER_STATE_PATH)
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
        spec = {**spec, "backbone": t_spec["backbone"]}
        model, groups = build_student(spec)
    else:
        model, groups = build_model_for(spec)
    trainable, total = count_params(model)
    proj_params = sum(p.numel() for p in model.proj.parameters())
    print(f"trainable {trainable:,} / {total:,} ({100*trainable/total:.2f}%) | of which projection head {proj_params:,}")

    history, train_secs = train_retriever(model, groups, spec, paths, teacher)
    load_trainable(model, paths["model_path"])
    if teacher is not None:
        del teacher; gc.collect(); torch.cuda.empty_cache()

    # final dev evaluation on the FULL dev pool: dense, TF-IDF, and the fused ranking
    t0 = time.time()
    dev_S, _ = dense_scores(model, dev_split)
    enc_secs = time.time() - t0
    dense_res = eval_scores(dev_split, dev_S)
    alpha, hyb_res = tune_hybrid(dev_split, dev_S, dev_split.tfidf_scores())
    print("FINAL DEV dense :", fmt(dense_res))
    print(f"FINAL DEV hybrid: {fmt(hyb_res)} (alpha {alpha})")

    test_S, _ = dense_scores(model, test_split)
    write_submission(paths["dense_dir"], predictions_from_scores(test_split, test_S), test_split)
    write_submission(paths["hybrid_dir"], predictions_from_scores(test_split, hybrid_scores(test_S, test_split.tfidf_scores(), alpha)), test_split)

    n_docs = len(dev_split.queries) + len(dev_split.pool_ids)
    info = {"experiment": name, "backbone": spec["backbone"], "technique": spec["technique"],
            "settings": {k: v for k, v in spec.items() if k != "name"}, "teacher": teacher_name,
            "trainable_params": trainable, "projection_params": proj_params, "adapter_params": trainable - proj_params,
            "total_params": total, "chunk": [CHUNK_LEN, MAX_CHUNKS], "pairs_per_epoch": PAIRS_PER_EPOCH,
            "hard_negatives": HARD_NEG, "tau": TAU, "epochs_run": len(history), "history": history,
            "dev": {"dense": dense_res, "hybrid": {**hyb_res, "alpha": alpha}},
            "train_seconds": round(train_secs), "dev_encode_seconds": round(enc_secs, 1),
            "docs_per_second": round(n_docs / enc_secs, 1),
            "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "checkpoint_mb": round(os.path.getsize(paths["model_path"]) / 1e6, 1)}
    with open(paths["run_info_path"], "w") as f:
        json.dump(info, f, indent=2)
    del model; gc.collect(); torch.cuda.empty_cache()
    return info

print("Runner defined.")

Runner defined.


In [23]:
def load_trainable(model, path):
    state = torch.load(path, map_location="cpu")
    res = model.load_state_dict(state, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    return model

SPECS = {
    "e1": dict(name="01_gqa_lorma", backbone="gqa", technique="lorma",
               targets=["q_proj", "k_proj", "v_proj", "c_proj"], r=8, alpha=16, lr=1e-4, lr_head=1e-3),
    "e5": dict(name="05_gqa_full_ft", backbone="gqa", technique="full", lr=1e-5, lr_head=1e-3),
    "e6": dict(name="06_kd_half_layer_student", backbone="gqa", technique="kd", lr=1e-4, lr_head=1e-3),
}
EXPERIMENT_SPECS = SPECS     # name used by the helper cell above

def load_dense_model(spec):
    """Rebuild a trained model and load its saved weights (this session's output, or an attached earlier output)."""
    if spec["technique"] == "kd":
        model = DocEncoder(build_student_lm(spec["backbone"])).to(device)
        return load_trainable(model, find_checkpoint(spec))
    return load_experiment_model(spec)

for k, s in SPECS.items():
    print(k, s["name"], "->", find_checkpoint(s))

e1 01_gqa_lorma -> /kaggle/input/datasets/himangshumahato/capstone-phase7-exp1-4-outputs/cs781_phase7/01_gqa_lorma/best_model.pt
e5 05_gqa_full_ft -> /kaggle/input/notebooks/himangshumahato/capstone-phase7-six-experiments/cs781_phase7/05_gqa_full_ft/best_model.pt
e6 06_kd_half_layer_student -> /kaggle/input/notebooks/himangshumahato/capstone-phase7-six-experiments/cs781_phase7/06_kd_half_layer_student/best_model.pt


## 6. Pool diagnostics (run this first and read it)

What matters for the tricks later:
- **queries in the pool**: if every query document is also in its own pool, the other queries are extra candidates that can never be returned for themselves.
- **gold that are query ids**: if this is 0, no query is ever the relevant answer for another query, so removing every query document from the candidates is free and removes distractors.
- **pool docs that are neither query nor gold**: if this is small, almost every remaining pool document is relevant to *some* query, which is what lets candidates "compete" across queries.

In [19]:
for split in (dev_split, test_split):
    qids = {q["id"] for q in split.queries}
    pool = set(split.pool_ids)
    line = f"{split.name:4s}: {len(qids)} queries | pool {len(pool)} | queries that are in the pool: {len(qids & pool)} ({len(qids & pool)/len(pool):.0%} of the pool)"
    if split.name != "test":
        gold = {r for q in split.queries for r in q["rel"]}
        line += (f"\n      unique gold {len(gold)} ({len(gold)/len(pool):.0%} of the pool) | gold that are query ids: {len(gold & qids)}"
                 f" | pool docs that are neither query nor gold: {len(pool - gold - qids)}")
    print(line)

dev : 118 queries | pool 1023 | queries that are in the pool: 118 (12% of the pool)
      unique gold 920 (90% of the pool) | gold that are query ids: 15 | pool docs that are neither query nor gold: 0
test: 237 queries | pool 1727 | queries that are in the pool: 237 (14% of the pool)


## 7. Lexical retrieval on the FULL case text

The earlier TF-IDF kept only the first 30,000 characters of each case; here `TEXT_LIMIT` is 300,000. Variants: TF-IDF unigrams, TF-IDF unigrams+bigrams, and BM25.
Each is shown plain and with every query document removed from the candidates.

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

def mask_query_cols(S, split, value=-1e9):
    """Remove every query document of the split from the candidates (columns of the pool that are query ids)."""
    qids = {q["id"] for q in split.queries}
    cols = [i for i, d in enumerate(split.pool_ids) if d in qids]
    S = S.copy(); S[:, cols] = value
    return S

def tfidf_matrix(split, ngram=(1, 1), min_df=2, max_features=400000):
    q_texts = [q["text"] for q in split.queries]
    vec = TfidfVectorizer(sublinear_tf=True, min_df=min_df, max_df=0.5, ngram_range=ngram, max_features=max_features, dtype=np.float32)
    vec.fit(split.pool_texts + q_texts)
    return (vec.transform(q_texts) @ vec.transform(split.pool_texts).T).toarray().astype(np.float32)

def bm25_matrix(split, k1=1.2, b=0.75, min_df=2, max_features=400000):
    q_texts = [q["text"] for q in split.queries]
    cv = CountVectorizer(min_df=min_df, max_df=0.5, max_features=max_features, dtype=np.float32)
    X = cv.fit_transform(split.pool_texts + q_texts).tocsr()
    n = len(split.pool_texts)
    Xp, Xq = X[:n].tocsr(), X[n:].tocsr()
    dl = np.asarray(Xp.sum(1)).ravel(); avgdl = dl.mean()
    df = np.asarray((Xp > 0).sum(0)).ravel()
    idf = np.log((n - df + 0.5) / (df + 0.5) + 1.0).astype(np.float32)
    rows = np.repeat(np.arange(n), np.diff(Xp.indptr))
    tf = Xp.data
    Xp.data = (tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / avgdl))).astype(np.float32)
    Xq.data = np.log1p(Xq.data)                     # dampened query term frequency
    return (Xq.multiply(idf) @ Xp.T).toarray().astype(np.float32)

LEX_VARIANTS = {
    "tfidf_1g":  lambda sp: tfidf_matrix(sp),
    "tfidf_12g": lambda sp: tfidf_matrix(sp, ngram=(1, 2), min_df=3, max_features=600000),
    "bm25_k1.2": lambda sp: bm25_matrix(sp, k1=1.2),
    "bm25_k2.0": lambda sp: bm25_matrix(sp, k1=2.0),
}
lex_dev = {}
for name, fn in LEX_VARIANTS.items():
    t0 = time.time(); lex_dev[name] = fn(dev_split)
    print(f"{name:10s} ({time.time()-t0:4.0f}s) plain: {fmt(eval_scores(dev_split, lex_dev[name]))}"
          f" | query docs removed: {fmt(eval_scores(dev_split, mask_query_cols(lex_dev[name], dev_split)))}")
print("reference (first 30,000 characters, unigram TF-IDF, from the earlier notebook): P@7 0.4334 | R@7 0.3679 | F1@7 0.3980")

tfidf_1g   (  11s) plain: P@7 0.3995 | R@7 0.3392 | F1@7 0.3669 | query docs removed: P@7 0.3935 | R@7 0.3340 | F1@7 0.3613
tfidf_12g  (  29s) plain: P@7 0.4419 | R@7 0.3751 | F1@7 0.4058 | query docs removed: P@7 0.4407 | R@7 0.3741 | F1@7 0.4047
bm25_k1.2  (   6s) plain: P@7 0.2252 | R@7 0.1912 | F1@7 0.2068 | query docs removed: P@7 0.2203 | R@7 0.1871 | F1@7 0.2023
bm25_k2.0  (   5s) plain: P@7 0.2591 | R@7 0.2199 | F1@7 0.2379 | query docs removed: P@7 0.2627 | R@7 0.2230 | F1@7 0.2412
reference (first 30,000 characters, unigram TF-IDF, from the earlier notebook): P@7 0.4334 | R@7 0.3679 | F1@7 0.3980


Pick the lexical variant with the best dev F1@7 (use the "query docs removed" column only if section 6 showed that no gold document is a query id):

In [21]:
LEX_CHOICE = max(lex_dev, key=lambda n: eval_scores(dev_split, lex_dev[n])["f1"])      # change by hand if you prefer another variant
print("lexical variant used from here on:", LEX_CHOICE)

lexical variant used from here on: tfidf_12g


## 8. Dense scores from the trained Phase 1 models, with more of each case

Each model is rebuilt from its saved weights and applied to up to `MAX_CHUNKS` chunks of every case (no retraining). Chunk embeddings are kept, so the effect of using the first 1, 2, 4 or 8 chunks is measured without re-encoding.
`DENSE_MODELS` lists the experiments to use (e1 = GQA + LoRMA, e6 = distilled student, e5 = GQA full fine-tuning); several models are averaged after row-wise z-scoring.

In [24]:
DENSE_MODELS = ["e1", "e5", "e6"]            # add "e5" for a third model (about 4 more minutes)

dense_parts = {}
for key in DENSE_MODELS:
    model = load_dense_model(SPECS[key])
    parts = {}
    for sp in (dev_split, test_split):
        t0 = time.time()
        q_emb, qc = embed_chunks_all(model, sp.q_chunks)
        c_emb, cc = embed_chunks_all(model, sp.pool_chunks)
        parts[sp.name] = (q_emb, qc, c_emb, cc)
        print(f"{key} {sp.name}: {len(sp.q_chunks)} queries + {len(sp.pool_chunks)} pool docs -> {len(c_emb)+len(q_emb)} chunks ({time.time()-t0:.0f}s)")
    dense_parts[key] = parts
    del model; gc.collect(); torch.cuda.empty_cache()

def agg_first_k(emb, counts, k):
    return F.normalize(torch.stack([p[:k].mean(0) for p in torch.split(emb, counts)]), dim=-1)

def dense_matrix(key, split, k):
    q_emb, qc, c_emb, cc = dense_parts[key][split.name]
    return (agg_first_k(q_emb, qc, k) @ agg_first_k(c_emb, cc, k).T).cpu().numpy()

for key in DENSE_MODELS: 
    for k in (1, 2, 4, 8):
        print(f"{key} first {k} chunk(s):", fmt(eval_scores(dev_split, dense_matrix(key, dev_split, k))))

[gqa] load_state_dict: <All keys matched successfully>


dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

dev pool:   0%|          | 0/1023 [00:00<?, ?it/s]

e1 dev: 118 queries + 1023 pool docs -> 8112 chunks (63s)


test queries:   0%|          | 0/237 [00:00<?, ?it/s]

test pool:   0%|          | 0/1727 [00:00<?, ?it/s]

e1 test: 237 queries + 1727 pool docs -> 13919 chunks (120s)
[gqa] load_state_dict: <All keys matched successfully>
e5 dev: 118 queries + 1023 pool docs -> 8112 chunks (66s)
e5 test: 237 queries + 1727 pool docs -> 13919 chunks (112s)
[student] load_state_dict: <All keys matched successfully>
e6 dev: 118 queries + 1023 pool docs -> 8112 chunks (34s)
e6 test: 237 queries + 1727 pool docs -> 13919 chunks (56s)
e1 first 1 chunk(s): P@7 0.2107 | R@7 0.1788 | F1@7 0.1934
e1 first 2 chunk(s): P@7 0.2470 | R@7 0.2097 | F1@7 0.2268
e1 first 4 chunk(s): P@7 0.2785 | R@7 0.2364 | F1@7 0.2557
e1 first 8 chunk(s): P@7 0.2809 | R@7 0.2384 | F1@7 0.2579
e5 first 1 chunk(s): P@7 0.1889 | R@7 0.1603 | F1@7 0.1734
e5 first 2 chunk(s): P@7 0.2458 | R@7 0.2086 | F1@7 0.2257
e5 first 4 chunk(s): P@7 0.2942 | R@7 0.2497 | F1@7 0.2702
e5 first 8 chunk(s): P@7 0.3123 | R@7 0.2652 | F1@7 0.2868
e6 first 1 chunk(s): P@7 0.2131 | R@7 0.1809 | F1@7 0.1957
e6 first 2 chunk(s): P@7 0.2494 | R@7 0.2117 | F1@7 0.229

## 9. Fusion and re-ranking, tuned on dev

Pipeline for a split: `H = w * z(dense) + (1 - w) * z(lexical)` (row-wise z-scores), then optionally
- **query removal**: every query document is dropped from the candidates,
- **pseudo-relevance feedback (PRF)**: add `beta * z(mean similarity to the current top-m documents)`; relevant candidates are usually cited together, so they resemble each other,
- **dual softmax** with temperature `T`: `log softmax over candidates + log softmax over queries`, so a candidate that fits another query much better is pushed down.

All settings are chosen by dev F1@7 (118 queries): keep the grid small and be wary of tiny differences.

In [25]:
def log_softmax_np(x, axis):
    m = x.max(axis=axis, keepdims=True)
    z = x - m
    return z - np.log(np.exp(z).sum(axis=axis, keepdims=True))

def dual_softmax(H, T):
    return log_softmax_np(H / T, 1) + log_softmax_np(H / T, 0)

def prf(H, docsim, m, beta, split, excl):
    Hm = mask_query_cols(H, split) if excl else H
    top = np.argsort(-Hm, axis=1)[:, :m]
    P = np.stack([docsim[t].mean(0) for t in top])
    return H + beta * zrow(P)

def components(split, k_chunks, lex_S):
    dense = np.mean([zrow(dense_matrix(key, split, k_chunks)) for key in DENSE_MODELS], axis=0)
    _, _, c_emb, cc = dense_parts[DENSE_MODELS[0]][split.name]
    C = agg_first_k(c_emb, cc, k_chunks)
    return {"dense": dense, "lex": zrow(lex_S), "docsim": zrow((C @ C.T).cpu().numpy())}

def pipeline(split, comps, cfg):
    H = cfg["w"] * comps["dense"] + (1 - cfg["w"]) * comps["lex"]
    if cfg["excl"]:
        H = mask_query_cols(H, split)
    if cfg["m"]:
        H = prf(H, comps["docsim"], cfg["m"], cfg["beta"], split, cfg["excl"])
        if cfg["excl"]:
            H = mask_query_cols(H, split)
    if cfg["T"]:
        H = dual_softmax(H, cfg["T"])
    return H

In [26]:
import itertools
K_CHUNKS = 8                          # chunks per case for the dense models (see the section 8 table)
dev_comps = components(dev_split, K_CHUNKS, lex_dev[LEX_CHOICE])

grid = [dict(w=w, excl=ex, T=T, m=m, beta=beta)
        for w, ex, T, (m, beta) in itertools.product((0.0, 0.2, 0.35, 0.5, 0.65, 0.8, 1.0), (False, True), (None, 0.5, 1.0, 2.0),
                                                     ((0, 0.0), (3, 0.3), (3, 0.6), (5, 0.3), (5, 0.6)))]
results = []
for cfg in grid:
    results.append((eval_scores(dev_split, pipeline(dev_split, dev_comps, cfg)), cfg))
results.sort(key=lambda r: -r[0]["f1"])
print(f"{len(grid)} configurations tried; top 10 by dev F1@7:")
for r, cfg in results[:10]:
    print(fmt(r), "|", cfg)
simple = [(r, c) for r, c in results if c["T"] is None]
print("\nbest WITHOUT the dual softmax:", fmt(simple[0][0]), "|", simple[0][1])
print("best lexical only            :", fmt(max((r for r, c in results if c["w"] == 0.0 and c["T"] is None and not c["excl"] and c["m"] == 0), key=lambda r: r["f1"])))
print("best dense only              :", fmt(max((r for r, c in results if c["w"] == 1.0 and c["T"] is None and not c["excl"] and c["m"] == 0), key=lambda r: r["f1"])))

280 configurations tried; top 10 by dev F1@7:
P@7 0.4831 | R@7 0.4101 | F1@7 0.4436 | {'w': 0.2, 'excl': False, 'T': 0.5, 'm': 3, 'beta': 0.3}
P@7 0.4818 | R@7 0.4090 | F1@7 0.4425 | {'w': 0.0, 'excl': False, 'T': 0.5, 'm': 3, 'beta': 0.6}
P@7 0.4818 | R@7 0.4090 | F1@7 0.4425 | {'w': 0.2, 'excl': True, 'T': 0.5, 'm': 3, 'beta': 0.3}
P@7 0.4806 | R@7 0.4080 | F1@7 0.4414 | {'w': 0.0, 'excl': False, 'T': 0.5, 'm': 3, 'beta': 0.3}
P@7 0.4806 | R@7 0.4080 | F1@7 0.4414 | {'w': 0.0, 'excl': True, 'T': 0.5, 'm': 5, 'beta': 0.6}
P@7 0.4806 | R@7 0.4080 | F1@7 0.4414 | {'w': 0.2, 'excl': False, 'T': 0.5, 'm': 5, 'beta': 0.3}
P@7 0.4806 | R@7 0.4080 | F1@7 0.4414 | {'w': 0.2, 'excl': True, 'T': 0.5, 'm': 5, 'beta': 0.3}
P@7 0.4794 | R@7 0.4070 | F1@7 0.4402 | {'w': 0.0, 'excl': True, 'T': 0.5, 'm': 5, 'beta': 0.3}
P@7 0.4794 | R@7 0.4070 | F1@7 0.4402 | {'w': 0.2, 'excl': False, 'T': 0.5, 'm': 0, 'beta': 0.0}
P@7 0.4794 | R@7 0.4070 | F1@7 0.4402 | {'w': 0.2, 'excl': False, 'T': 0.5, 'm': 5, '

## 10. Test submissions

`rerank_full/submission.zip` uses the best configuration overall; `rerank_simple/submission.zip` uses the best configuration without the dual softmax (it does not use the other test queries for normalisation).
Both pass the format checks (7 unique candidates per query, all inside the test pool, every query covered).

In [27]:
test_comps = components(test_split, K_CHUNKS, LEX_VARIANTS[LEX_CHOICE](test_split))
for tag, (res, cfg) in (("full", results[0]), ("simple", simple[0])):
    H = pipeline(test_split, test_comps, cfg)
    preds = predictions_from_scores(test_split, H)
    print(f"{tag}: dev {fmt(res)} | config {cfg}")
    write_submission(os.path.join(BASE_OUTPUT_DIR, f"rerank_{tag}"), preds, test_split)

full: dev P@7 0.4831 | R@7 0.4101 | F1@7 0.4436 | config {'w': 0.2, 'excl': False, 'T': 0.5, 'm': 3, 'beta': 0.3}
saved: /kaggle/working/cs781_phase7_rerank/rerank_full/submission.zip | queries: 237
simple: dev P@7 0.4504 | R@7 0.3823 | F1@7 0.4136 | config {'w': 0.35, 'excl': False, 'T': None, 'm': 5, 'beta': 0.6}
saved: /kaggle/working/cs781_phase7_rerank/rerank_simple/submission.zip | queries: 237


## Notes for the report
- Report the pool diagnostics (section 6), the lexical table (7), the chunk-coverage table (8) and the top of the grid (9): they answer the long-document, representation and negative-handling questions.
- State which steps use the other test queries (dual softmax) and which depend on the pool structure (query removal); give the `simple` submission's score next to the `full` one.
- The dense models are still the Phase 1 models adapted in experiments 1, 5 and 6; the lexical model and the fusion are extra ranking components, so describe them as such.
- Dev F1@7 is measured on 118 queries and the grid is tuned on it, so expect the Codabench score to be lower than the dev score.